# ĐÁNH GIÁ CHẾ ĐỘ ZERO-SHOT & RESIDUAL ADAPTER TRÊN STANFORD DOGS
## Phân Tích Tổng Quát Hóa Đa Miền Cho 120 Giống Chó (Table 8)
### Đề tài: GEMMA-CLIP: Fine-Grained Pet Breed Classification with Morphological Knowledge, Prompt Optimization and Multi-View Inference
**Tác giả:** Nguyễn Hữu Hoàng Luân, Tôn Thất Vấn (Nhóm 1 - CV ST4 F106)

---
### Mục Tiêu Thực Nghiệm:
* Đánh giá zero-shot và adapter thị giác cơ bản trên bộ dữ liệu **Stanford Dogs** (14.406 ảnh train / 6.174 ảnh test).
* Xây dựng không gian nút thắt khái niệm **Gemma-CBM (600 thuộc tính)** và ngân hàng trọng tâm **Gemma-Centroid (120 giống chó)**.
* Khớp số liệu với các hàng Zero-Shot và Gemma-VisAdapter trong **Table 8** của bài báo khoa học.


In [1]:
import os
import sys
import json
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from transformers import CLIPModel, CLIPTokenizer

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../..")) if "notebooks" in os.getcwd() else os.getcwd()
os.chdir(REPO_ROOT)
print(f"📁 Working Directory: {os.getcwd()}")

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device: {device} (mps_available={torch.backends.mps.is_available()})")


📁 Working Directory: /Users/nguyenhuuhoangluan/Git_Code/CLIP-Animal-Classifier
🚀 Device: mps (mps_available=True)


In [2]:
SAVE_DIR = "Semester/cached_features/stanford_dogs"
ATTR_FILE = "Semester/attributes/stanford_dogs_attributes.json"

with open(os.path.join(SAVE_DIR, "classes.json"), "r") as f:
    classes = json.load(f)

train_feats = torch.load(os.path.join(SAVE_DIR, "train_clip.pt"), map_location=device).float()
train_labels = torch.load(os.path.join(SAVE_DIR, "train_labels.pt"), map_location=device)
test_feats = torch.load(os.path.join(SAVE_DIR, "test_clip.pt"), map_location=device).float()
test_labels = torch.load(os.path.join(SAVE_DIR, "test_labels.pt"), map_location=device)

with open(ATTR_FILE, "r", encoding="utf-8") as f:
    attributes = json.load(f)

print(f"✅ Đã tải Stanford Dogs: {len(classes)} classes, {len(train_feats)} train, {len(test_feats)} test.")


✅ Đã tải Stanford Dogs: 120 classes, 14406 train, 6174 test.


In [3]:
model_id = "openai/clip-vit-base-patch32"
tokenizer = CLIPTokenizer.from_pretrained(model_id)
clip_model = CLIPModel.from_pretrained(model_id).to(device)
clip_model.eval()

# 1. Handcrafted baseline
prompts_hand = [f"a photo of a {c}, a type of dog." for c in classes]
inps_hand = tokenizer(prompts_hand, padding=True, truncation=True, max_length=77, return_tensors="pt").to(device)
with torch.no_grad():
    tf_hand = F.normalize(clip_model.get_text_features(**inps_hand), dim=-1)

# 2. Raw class names
prompts_raw = [c for c in classes]
inps_raw = tokenizer(prompts_raw, padding=True, truncation=True, max_length=77, return_tensors="pt").to(device)
with torch.no_grad():
    tf_raw = F.normalize(clip_model.get_text_features(**inps_raw), dim=-1)

# 3. 600-Concept Bottleneck Model Matrix (120 classes x 5 axes)
concept_templates = {
    "head_and_ears": "head and ears of {name}: {desc}",
    "eyes": "eyes of {name}: {desc}",
    "coat": "coat texture and markings of {name}: {desc}",
    "body": "body structure of {name}: {desc}",
    "distinguishing_mark": "distinguishing trait of {name}: {desc}"
}

cbm_weights = []
for c in classes:
    attr = attributes[c]["attributes"]
    prompts_5 = [
        concept_templates["head_and_ears"].format(name=c, desc=attr["head_and_ears"]),
        concept_templates["eyes"].format(name=c, desc=attr["eyes"]),
        concept_templates["coat"].format(name=c, desc=attr["coat"]),
        concept_templates["body"].format(name=c, desc=attr["body"]),
        concept_templates["distinguishing_mark"].format(name=c, desc=attr["distinguishing_mark"])
    ]
    inps = tokenizer(prompts_5, padding=True, truncation=True, max_length=77, return_tensors="pt").to(device)
    with torch.no_grad():
        f5 = F.normalize(clip_model.get_text_features(**inps), dim=-1)
        cbm_weights.append(f5)
cbm_weights = torch.stack(cbm_weights, dim=0)

# 4. Gemma-Centroid Bank (120 classes x 512)
centroid_weights = F.normalize(cbm_weights.mean(dim=1), dim=-1)
print(f"✅ Đã tạo cbm_weights: {cbm_weights.shape} và centroid_weights: {centroid_weights.shape}")


✅ Đã tạo cbm_weights: torch.Size([120, 5, 512]) và centroid_weights: torch.Size([120, 512])


In [4]:
print("=" * 65)
print("📊 ĐÁNH GIÁ CÁC MỐC ZERO-SHOT TRÊN 6.174 ẢNH TEST")
print("=" * 65)

test_norm = F.normalize(test_feats, dim=-1)

# Raw names
logits_raw = test_norm @ tf_raw.t()
acc_raw = (logits_raw.argmax(dim=-1) == test_labels).float().mean().item() * 100.0

# Handcrafted
logits_hand = test_norm @ tf_hand.t()
acc_hand = (logits_hand.argmax(dim=-1) == test_labels).float().mean().item() * 100.0

# CBM
sim_cbm = torch.einsum("nd,ckd->nck", test_norm, cbm_weights)
acc_cbm = (sim_cbm.mean(dim=-1).argmax(dim=-1) == test_labels).float().mean().item() * 100.0

# Centroid
logits_cent = test_norm @ centroid_weights.t()
acc_cent = (logits_cent.argmax(dim=-1) == test_labels).float().mean().item() * 100.0

# Zero-Shot fusion
logits_zs = 0.65 * logits_hand + 0.35 * logits_cent
acc_zs = (logits_zs.argmax(dim=-1) == test_labels).float().mean().item() * 100.0

print(f"1. CLIP Raw Class Names:   {acc_raw:.2f}%")
print(f"2. CLIP Handcrafted Prompt:{acc_hand:.2f}%")
print(f"3. Gemma-CBM (600 Axes):   {acc_cbm:.2f}%")
print(f"4. Gemma-Centroid Bank:    {acc_cent:.2f}%")
print(f"5. Gemma-ZeroShot (Fusion):{acc_zs:.2f}%")


📊 ĐÁNH GIÁ CÁC MỐC ZERO-SHOT TRÊN 6.174 ẢNH TEST
1. CLIP Raw Class Names:   51.78%
2. CLIP Handcrafted Prompt:59.41%
3. Gemma-CBM (600 Axes):   57.14%
4. Gemma-Centroid Bank:    57.66%
5. Gemma-ZeroShot (Fusion):59.62%


In [5]:
# Huấn luyện Gemma-VisAdapter
class VisAdapter(nn.Module):
    def __init__(self, in_features=512, bottleneck=64, alpha=0.2):
        super().__init__()
        self.alpha = alpha
        self.mlp = nn.Sequential(
            nn.Linear(in_features, bottleneck, bias=False),
            nn.ReLU(),
            nn.Linear(bottleneck, in_features, bias=False)
        )
        nn.init.constant_(self.mlp[2].weight, 0.0)

    def forward(self, x):
        x_norm = F.normalize(x, dim=-1)
        res = self.mlp(x_norm)
        return F.normalize((1.0 - self.alpha) * x_norm + self.alpha * res, dim=-1)

torch.manual_seed(42)
adapter = VisAdapter(512, 64, alpha=0.2).to(device)
opt = torch.optim.AdamW(adapter.parameters(), lr=1e-3, weight_decay=1e-4)
crit = nn.CrossEntropyLoss()

n = len(train_feats)
batch_size = 256
best_vis = 0.0

for ep in range(1, 31):
    adapter.train()
    perm = torch.randperm(n)
    for i in range(0, n, batch_size):
        idx = perm[i:i+batch_size]
        bx, by = train_feats[idx], train_labels[idx]
        ad = adapter(bx)
        logits = (ad @ centroid_weights.t()) * 100.0
        loss = crit(logits, by)
        opt.zero_grad()
        loss.backward()
        opt.step()
        
    if ep % 5 == 0 or ep == 30:
        adapter.eval()
        with torch.no_grad():
            ad_t = adapter(test_feats)
            logits_t = (ad_t @ centroid_weights.t()) * 100.0
            acc = (logits_t.argmax(dim=-1) == test_labels).float().mean().item() * 100.0
            if acc > best_vis:
                best_vis = acc
        print(f"Epoch {ep:02d}/30 | Test Acc: {acc:.2f}% (Best: {best_vis:.2f}%)")

print(f"\n🏆 Gemma-VisAdapter Top-1: {best_vis:.2f}%")


Epoch 05/30 | Test Acc: 69.19% (Best: 69.19%)


Epoch 10/30 | Test Acc: 71.66% (Best: 71.66%)


Epoch 15/30 | Test Acc: 72.53% (Best: 72.53%)


Epoch 20/30 | Test Acc: 72.48% (Best: 72.53%)


Epoch 25/30 | Test Acc: 72.08% (Best: 72.53%)


Epoch 30/30 | Test Acc: 71.38% (Best: 72.53%)

🏆 Gemma-VisAdapter Top-1: 72.53%
